# JAX-CrossCat Local Test Runner

Run the test suite locally on a GTX 1650 (4 GB VRAM).

**Usage:** Open in VS Code or Jupyter, select your Python kernel, and run all cells.

## 1. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
import jax

print(f"JAX version: {jax.__version__}")
print(f"Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

assert jax.default_backend() == "gpu", "GPU not detected — check your JAX CUDA install"

import crosscat

print(f"jax-crosscat version: {crosscat.__version__}")

## 2. Run fast tests

In [ ]:
import os
import subprocess

os.chdir(
    os.path.join(
        os.path.dirname(os.getcwd())
        if os.path.basename(os.getcwd()) == "notebooks"
        else os.getcwd()
    )
)

# Ensure we're at repo root
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)
print(f"Repo root: {repo_root}")

In [ ]:
!uv sync --extra dev --extra gpu

In [ ]:
!python -m pytest -m "not slow" -v --tb=short --timeout=300 2>&1

## 3. Run slow tests (optional)

These include integration/recovery tests with 30+ Gibbs sweeps. May take 30-60 min on GTX 1650.

In [ ]:
!python -m pytest -m slow -v --tb=short --timeout=600 2>&1

## 4. Lint check

In [ ]:
!ruff check crosscat/ tests/ && echo "Lint passed" || echo "Lint failed"
!ruff format --check crosscat/ tests/ && echo "Format OK" || echo "Format issues"

## 5. Smoke test — end-to-end inference

In [ ]:
import time

import jax

from crosscat import (
    column_partition_ari,
    generate_crosscat_data,
    initialize,
    log_joint,
    pack_state,
    packed_gibbs_sweep,
    unpack_state,
)
from crosscat.packed.aot_cache import enable_xla_cache
from crosscat.types import ColumnType

enable_xla_cache()

key = jax.random.key(42)
result = generate_crosscat_data(
    key,
    n_rows=100,
    column_types=[
        ColumnType.CONTINUOUS,
        ColumnType.CONTINUOUS,
        ColumnType.CATEGORICAL,
        ColumnType.BINARY,
    ],
    n_views=2,
    n_clusters=3,
)
data = result["data"]
col_types = result["column_types"]
true_col_assigns = result["true_column_assignments"]

print(f"Data shape: {data.shape}")
print(f"Column types: {col_types}")

In [ ]:
key, k1, k2 = jax.random.split(key, 3)
state = initialize(k1, data, col_types)
packed = pack_state(state, max_views=8, max_clusters=16)

print("Running packed Gibbs sweep (first call triggers JIT compilation)...")
t0 = time.time()
packed = packed_gibbs_sweep(k2, packed, data, n_sweeps=20)
t1 = time.time()
print(f"20 sweeps in {t1 - t0:.1f}s (includes JIT compilation)")

key, k3 = jax.random.split(key)
t0 = time.time()
packed = packed_gibbs_sweep(k3, packed, data, n_sweeps=20)
t1 = time.time()
print(f"20 more sweeps in {t1 - t0:.1f}s (compiled)")

state = unpack_state(packed, col_types, data=data)
score = log_joint(state, data)
ari = column_partition_ari(state, true_col_assigns)
print(f"\nLog joint: {score:.2f}")
print(f"Column partition ARI: {ari:.3f}")
print(f"Discovered {state.n_views} views")
print("\nSmoke test passed!")